# Feature Engineering

## Objectif

Ce notebook a pour objectif d'appliquer le feature engineering aux données après traitement des outliers. Il crée de nouvelles features, encode les variables catégorielles, et transforme les features asymétriques.

**Principales étapes** :
- Création de nouvelles features (TotalSF, TotalBathrooms, HouseAge, etc.)
- Encodage des variables catégorielles (ordinal pour les variables de qualité, label encoding pour les autres)
- Transformation des features asymétriques (log transformation pour les features avec skewness > 0.75)
- Création de la variable cible transformée (SalePrice_log) si nécessaire
- Sauvegarde des données avec features créées

## Prérequis

**À exécuter AVANT ce notebook** :

1. **`exploration_base_donnees.ipynb`** - Toutes les sections (1-15)
   - Ce notebook doit être exécuté en entier pour générer :
     - `data/processed/train_outliers_treated.csv`
     - `data/processed/test_outliers_treated.csv`

**Fichiers d'entrée requis** :
- `data/processed/train_outliers_treated.csv` - Données d'entraînement après traitement des outliers
- `data/processed/test_outliers_treated.csv` - Données de test après traitement des outliers

**Fichiers générés** :
- `data/processed/train_features.csv` - Données d'entraînement avec features créées
- `data/processed/test_features.csv` - Données de test avec features créées

In [ ]:
# Imports
import sys
from pathlib import Path

# Ajouter le dossier parent au path pour permettre l'import des modules du projet
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Import des bibliothèques standards pour la manipulation de données
import pandas as pd
import numpy as np
import logging

# Import de la classe FeatureEngineer qui contient les méthodes de feature engineering
from src.feature_engineering import FeatureEngineer

# Configuration du niveau de logging pour afficher les messages informatifs
logging.basicConfig(level=logging.INFO)

In [ ]:
# Chargement des données après traitement des outliers
# Les fichiers train_outliers_treated.csv et test_outliers_treated.csv sont générés par exploration_base_donnees.ipynb
train_df = pd.read_csv("../data/processed/train_outliers_treated.csv", keep_default_na=False, na_filter=False)
test_df = pd.read_csv("../data/processed/test_outliers_treated.csv", keep_default_na=False, na_filter=False)

print("=" * 60)
print("CHARGEMENT DES DONNÉES (APRÈS TRAITEMENT DES OUTLIERS)")
print("=" * 60)
print(f"Données chargées: Train {train_df.shape}, Test {test_df.shape}")

# Initialisation de la classe FeatureEngineer qui contient les méthodes de transformation
feature_engineer = FeatureEngineer()

# Création de nouvelles features composites à partir des variables existantes
# Exemples: TotalSF, TotalBathrooms, HouseAge, etc.
train_df = feature_engineer.create_features(train_df)
test_df = feature_engineer.create_features(test_df)

print(f"\nAprès création de features: Train {train_df.shape}, Test {test_df.shape}")

# Identification des colonnes catégorielles (type object) et numériques (type number)
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
# Exclusion de SalePrice des colonnes numériques car c'est la variable cible
numeric_cols = [col for col in train_df.select_dtypes(include=[np.number]).columns.tolist() 
                if col != 'SalePrice']

print(f"\nColonnes catégorielles: {len(categorical_cols)}")
print(f"Colonnes numériques: {len(numeric_cols)}")

# Encodage des variables catégorielles en valeurs numériques
# Encodage ordinal pour les variables de qualité (Ex > Gd > TA > Fa > Po)
# Label encoding pour les autres variables catégorielles
train_df = feature_engineer.encode_categorical(train_df, categorical_cols, encoding_type='ordinal')
test_df = feature_engineer.encode_categorical(test_df, categorical_cols, encoding_type='ordinal')

# Transformation logarithmique des features asymétriques (skewness > 0.75)
# Cette transformation réduit l'asymétrie et améliore la distribution des données
train_df = feature_engineer.transform_skewed_features(train_df, numeric_cols, threshold=0.75)
test_df = feature_engineer.transform_skewed_features(test_df, numeric_cols, threshold=0.75)

# Création de la variable cible transformée SalePrice_log à partir de SalePrice
# La transformation logarithmique (log1p) normalise la distribution de la variable cible
if 'SalePrice' in train_df.columns and 'SalePrice_log' not in train_df.columns:
    print("\nCréation de SalePrice_log à partir de SalePrice...")
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    print(f"   SalePrice_log créé. Skewness: {train_df['SalePrice_log'].skew():.3f}")

# Sauvegarde des données transformées avec toutes les features créées
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_feat_path = output_dir / "train_features.csv"
test_feat_path = output_dir / "test_features.csv"

train_df.to_csv(train_feat_path, index=False)
test_df.to_csv(test_feat_path, index=False)

print(f"\n{'='*60}")
print("SAUVEGARDE DES DONNÉES AVEC FEATURES")
print(f"{'='*60}")
print(f"\nDonnées avec features sauvegardées:")
print(f"   - Train: {train_feat_path} ({train_df.shape[0]} observations, {train_df.shape[1]} colonnes)")
print(f"   - Test: {test_feat_path} ({test_df.shape[0]} observations, {test_df.shape[1]} colonnes)")
print(f"\nLes données sont prêtes pour l'entraînement des modèles dans modeling.ipynb")
